In [ ]:
! pip install langchain langchain-openai langchain-huggingface langchain-community langchain-text-splitters faiss-cpu python-dotenv
! pip install pymupdf neo4j sentence-transformers
! pip install ragas datasets

In [ ]:
import os
import sys

# Adiciona o diretório do projeto ao path para importar src/
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from datasets import Dataset

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

from src.knowledge_graph import KnowledgeGraph
from src.chatbot import Chatbot
from src.ingestion import load_or_create_index

In [ ]:
load_dotenv()

In [ ]:
test_queries = [
    "O que é lógica proposicional segundo a apostila?",
    "Como a apostila define uma proposição?",
    "O que são conectivos lógicos e quais são apresentados no material?",
    "O que é uma tabela-verdade e para que ela é utilizada?",
    "Como a apostila define tautologia, contradição e contingência?"
]

ground_truths = [
    "Lógica proposicional é o ramo da lógica que estuda proposições e as relações entre elas por meio de conectivos lógicos.",
    "Proposição é toda sentença declarativa que pode ser classificada como verdadeira ou falsa, mas não ambas.",
    "Conectivos lógicos são operadores que conectam proposições, como negação (¬), conjunção (∧), disjunção (∨), condicional (→) e bicondicional (↔).",
    "Tabela-verdade é um método utilizado para determinar o valor lógico de proposições compostas a partir dos valores lógicos das proposições simples.",
    "Tautologia é uma proposição composta que é sempre verdadeira; contradição é sempre falsa; contingência é aquela que pode ser verdadeira ou falsa dependendo dos valores das proposições componentes."
]

In [ ]:
# Garante que o índice FAISS existe (cria a partir de ../docs/ se necessário)
print("Inicializando índice FAISS...")
load_or_create_index()

In [ ]:
# Knowledge Graph (Neo4j) — falha de forma graceful se não disponível
try:
    knowledge_graph = KnowledgeGraph()
    print("Knowledge Graph conectado.")
except Exception as e:
    print(f"Aviso: Knowledge Graph indisponível ({e}). Continuando sem KG.")
    knowledge_graph = None

chatbot = Chatbot(knowledge_graph=knowledge_graph)
print("Chatbot inicializado.")

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"},
)

In [ ]:
print("Coletando respostas para avaliação RAGAS...
")
ragas_data = []

for i, query in enumerate(test_queries):
    print(f"  [{i+1}/{len(test_queries)}] {query}")

    retrieval = chatbot.retriever.retrieve(query)
    contexts = [doc.page_content for doc in retrieval["docs"]]

    chat_result = chatbot.chat(query)
    answer = chat_result["answer"]

    ragas_data.append({
        "question": query,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": ground_truths[i]
    })

In [ ]:
eval_llm = ChatOpenAI(
    model="llama3.3-70b-instruct",
    openai_api_key=os.getenv("OPENAI_API_KEY", ""),
    openai_api_base="https://inference.do-ai.run/v1",
    temperature=0,
)

dataset = Dataset.from_list(ragas_data)

print("Executando avaliação RAGAS...")
result = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=eval_llm,
    embeddings=embeddings,
)

print("
=== RESULTADOS RAGAS ===")
print(result)

df = result.to_pandas()
print("
Detalhes por query:")
print(df.to_string())